# **Phase 4 - Fine-tune Reranker**

In [1]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cu121
%pip install -qU "transformers<5.0.0" FlagEmbedding accelerate 
%pip install -qU joblib faiss-cpu bm25s

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
flagembedding 1.3.5 requires transformers>=4.44.2, which is not installed.
peft 0.19.1 requires transformers, which is not installed.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES']    = '0'
os.environ['NCCL_DEBUG']              = 'WARN'
os.environ['NCCL_IB_DISABLE']         = '1'
os.environ['NCCL_P2P_DISABLE']        = '1'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
os.environ['OMP_NUM_THREADS']         = '1'
os.environ['TORCH_CPP_LOG_LEVEL']     = 'ERROR'
os.environ['PYTHONWARNINGS']          = 'ignore::FutureWarning,ignore::UserWarning'

import gc
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from joblib import Parallel, delayed

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

import faiss
import bm25s

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## **1. Load Data**

In [3]:
DATA_DIR      = Path('workspace/data/cleaned')
PROCESSED_DIR = Path('workspace/data/processed')

FINETUNE_DATA_DIR = Path('workspace/data/finetune')
FINETUNE_DATA     = Path('workspace/data/finetune/ft_reranker_data.jsonl')

FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('workspace/models/reranker')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = MODEL_DIR / 'cache'
(CACHE_DIR / 'model').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'data').mkdir(parents=True, exist_ok=True)

In [4]:
corpus      = pd.read_parquet(DATA_DIR / 'corpus.parquet')
train_split = pd.read_parquet(DATA_DIR / 'train_split.parquet')

corpus       = corpus.reset_index(drop=True)
corpus_texts = corpus['text'].tolist()
corpus_cids  = corpus['cid'].tolist()

cid2text = dict(zip(corpus_cids, corpus_texts))
cid2idx  = {cid: idx for idx, cid in enumerate(corpus_cids)}
idx2cid  = {idx: cid for cid, idx in cid2idx.items()}

print(len(corpus), len(train_split))

236199 96453


## **2. Load Retrieval Artifacts**

In [5]:
corpus_tokens = pd.read_parquet(PROCESSED_DIR / 'bm25_corpus_tokens.parquet')['tokens'].tolist()
query_tokens  = pd.read_parquet(PROCESSED_DIR / 'bm25_train_tokens.parquet')['tokens'].tolist()

corpus_tokens = [list(map(str, x)) for x in corpus_tokens]
query_tokens  = [list(map(str, x)) for x in query_tokens]

bm25 = bm25s.BM25(k1=1.5, b=0.75)
bm25.index(corpus_tokens)

BM25S Create Vocab:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/236199 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/236199 [00:00<?, ?it/s]

In [6]:
query_embeds = np.load(PROCESSED_DIR / 'ft_train_embeddings.npy', mmap_mode='r')
faiss_index  = faiss.read_index(str(PROCESSED_DIR / 'ft_faiss_index.bin'))

## **3. Candidate Retrieval (Dense + BM25)**

In [7]:
CANDIDATE_K = 30

def idx_to_cids(matrix):
    return [[idx2cid[i] for i in row if i >= 0] for row in matrix]

_, dense_indices = faiss_index.search(query_embeds.astype(np.float32), CANDIDATE_K)

dense_cands   = idx_to_cids(dense_indices)
bm25_cands, _ = bm25.retrieve(query_tokens, k=CANDIDATE_K, n_threads=-1)

BM25S Retrieve:   0%|          | 0/96453 [00:00<?, ?it/s]

## **4. Build Training Data (Hard Negative Mining)**

In [8]:
# normalize cid
train_split['cid'] = train_split['cid'].apply(
    lambda x: [int(i) for i in (x.tolist() if isinstance(x, np.ndarray) else x)]
)

# convert candidates -> int
dense_cands = [[int(x) for x in row] for row in dense_cands]
bm25_cands  = [[int(x) for x in row] for row in bm25_cands]

# negative samples per query
NEG_COUNT   = 7

In [9]:
def fast_mine_negatives(pos_set, dense, bm25_list):
    candidates = []

    for c in dense[:50]:
        if c not in pos_set:
            candidates.append(c)
            if len(candidates) == NEG_COUNT:
                return candidates

    for c in bm25_list[:50]:
        if c not in pos_set:
            candidates.append(c)
            if len(candidates) == NEG_COUNT:
                return candidates

    return candidates

In [10]:
written = 0

with open(FINETUNE_DATA, 'w', encoding='utf-8') as f:
    for i in tqdm(range(len(train_split))):
        row      = train_split.iloc[i]
        query    = row.question
        pos_cids = row.cid
        pos_set  = set(pos_cids)

        pos_texts = [cid2text[c] for c in pos_cids if c in cid2text]
        if not pos_texts:
            continue

        neg_ids   = fast_mine_negatives(pos_set, dense_cands[i], bm25_cands[i])
        neg_texts = [cid2text[c] for c in neg_ids]

        record = {"query": query, "pos": pos_texts[:3], "neg": neg_texts}
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        written += 1

print("Written:", written)

  0%|          | 0/96453 [00:00<?, ?it/s]

Written: 96453


## **5. Finetune Reranker**

In [23]:
gc.collect()
torch.cuda.empty_cache()

In [24]:
!torchrun --standalone --nproc_per_node=1 \
-m FlagEmbedding.finetune.reranker.encoder_only.base \
--model_name_or_path BAAI/bge-reranker-v2-m3 \
--cache_dir {CACHE_DIR}/model \
--train_data {FINETUNE_DATA} \
--cache_path {CACHE_DIR}/data \
--output_dir {MODEL_DIR} \
--overwrite_output_dir \
--train_group_size {1 + NEG_COUNT} \
--query_max_len 512 \
--passage_max_len 512 \
--max_len 512 \
--pad_to_multiple_of 8 \
--per_device_train_batch_size 4 \
--gradient_accumulation_steps 8 \
--learning_rate 1e-5 \
--num_train_epochs 1 \
--warmup_ratio 0.1 \
--weight_decay 0.01 \
--logging_steps 20 \
--fp16 False \
--bf16 True \
--gradient_checkpointing True \
--torch_compile True \
--dataloader_num_workers 8 \
--dataloader_pin_memory True \
--ddp_find_unused_parameters False \
--save_strategy no \
--report_to none

04/22/2026 06:20:26 - WARNING - FlagEmbedding.abc.finetune.reranker.AbsRunner -   Process rank: 0, device: cuda:0, n_gpu: 1, distributed training: True, 16-bits training: False
04/22/2026 06:20:26 - INFO - FlagEmbedding.abc.finetune.reranker.AbsRunner -   Training/evaluation parameters AbsRerankerTrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=8,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters

In [27]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path    = 'workspace/models/reranker',
    repo_id        = 'YuITC/bge-reranker-v2-m3-vn-legal',
    repo_type      = 'model',
    commit_message = 'update: retrain with larger batch size and data',
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/hf_api.py:4205: UserWarning: It seems that you are about to commit a data file (cache/data/json/default-7949963fb415936c/0.0.0/95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2/json-train-00000-of-00003.arrow) to a model repository. You are sure this is intended? If you are trying to upload a dataset, please set `repo_type='dataset'` or `--repo-type=dataset` in a CLI.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/hf_api.py:4205: UserWarning: It seems that you are about to commit a data file (cache/data/json/default-7949963fb415936c/0.0.0/95cd12cb79bdd40ece813ca76cc5f709067bfb719d56048dc7651d7403a4c3d2/json-train-00001-of-00003.arrow) to a model repository. You are sure this is intended? If you are trying to upload a dataset, please set `repo_type='dataset'` or `--repo-type=dataset` in a CLI.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/hf_api.py:4205: UserWarnin

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/YuITC/bge-reranker-v2-m3-vn-legal/commit/27012120132bcb3e66f21b115fa08649975a2efe', commit_message='update: retrain with larger batch size and data', commit_description='', oid='27012120132bcb3e66f21b115fa08649975a2efe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/YuITC/bge-reranker-v2-m3-vn-legal', endpoint='https://huggingface.co', repo_type='model', repo_id='YuITC/bge-reranker-v2-m3-vn-legal'), pr_revision=None, pr_num=None)